In [ ]:
from pprint import pprint

In [ ]:
from transformers import AutoTokenizer

# Load the tokenizer that ships WITH the model checkpoint.
# Using a mismatched tokenizer produces wrong token IDs and broken training.
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

In [ ]:
text = "Fine-tuning is fun!"

In [ ]:
# Token ID
print(tok(text))

In [ ]:
print(tok.convert_ids_to_tokens(tok(text)["input_ids"]))  # token IDs -> readable pieces
# Ký tự Ġ không phải là chữ "G" thật – đó là ký tự Unicode U+0120 ("LATIN CAPITAL LETTER G WITH DOT ABOVE"), 
# được các tokenizer kiểu BPE byte-level (GPT-2, GPT-3, RoBERTa,...) dùng làm ký hiệu thay thế cho dấu cách (space).

In [ ]:
# Decoded back
print("Decoded back:", tok.decode(tok(text)["input_ids"]))

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Who won the world series in 2020?"},
    {"role": "assistant", "content": "The Los Angeles Dodgers won the World Series in 2020."},
    {"role": "user", "content": "Where was it played?"}
]

In [ ]:
formatted = tok.apply_chat_template(
    messages,
    tokenize=False, # return a string first so we can inspect it
    add_generation_prompt=True, # False for TRAINING (the reply is already given)
)

In [ ]:
print(formatted)

In [ ]:
train_msgs = [
    [
        {"role": "user", "content": "Explain QLoRA in one sentence."},
        {"role": "assistant", "content": "QLoRA is a fine-tuning method that combines 4-bit quantization of a pretrained model with low-rank adapters (LoRA), enabling large language models to be fine-tuned efficiently on limited GPU memory."}
    ],
    [
        {"role": "user", "content": "What is the main benefit of LoRA compared to full fine-tuning?"},
        {"role": "assistant", "content": "LoRA drastically reduces the number of trainable parameters by only updating small low-rank matrices instead of the full model weights, saving memory and compute while preserving performance."}
    ],
    [
        {"role": "user", "content": "Why do we quantize the base model in QLoRA?"},
        {"role": "assistant", "content": "Quantizing the base model to 4-bit precision shrinks its memory footprint significantly, allowing larger models to fit on a single GPU while the LoRA adapters are still trained in higher precision for accuracy."}
    ],
]
prompt = tok.apply_chat_template(
    train_msgs,
    tokenize=False,
    add_generation_prompt=False   # True: open the assistant turn for generation
)
pprint(prompt)

In [ ]:
print(tok.pad)

# Ráp tất cả trong bài 5

In [ ]:
!pip install -U "huggingface_hub"

In [ ]:
!pip install trl

In [ ]:
!pip install -U "bitsandbytes>=0.46.1"

In [ ]:
# import os
# os.environ["WANDB_DISABLED"] = "true"
# os.environ["WANDB_MODE"] = "disabled"

In [ ]:
from huggingface_hub import login
from getpass import getpass

token = getpass("Nhập HF token: ")
login(token)

In [ ]:
from datasets import load_dataset

# Mỗi row có 1 field "messages" chứa list dict role/content
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")

In [ ]:
def build_training_prompt(data, tokenizer):
    prompt = tokenizer.apply_chat_template(
        data,
        tokenize=False,
        add_generation_prompt=False
    )
    return prompt

In [ ]:
from transformers.utils import is_bitsandbytes_available
print(is_bitsandbytes_available())

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import torch
import bitsandbytes as bnb

# 4-bit quantization to fit the base model into limited VRAM (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [ ]:
model_id = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
ds = [x["messages"] for x in dataset]
ds = list(map(lambda x: build_training_prompt(x, tokenizer), ds))
split_dataset = ds.train_test_split(test_size=0.1, shuffle=True, seed=42)
train_ds = split_dataset["train"]
test_ds = split_dataset["test"]

In [ ]:
# LoRA adapter config (rank / alpha explained in Week 4)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

# The three key hyperparameters live here
sft_config = SFTConfig(
    output_dir="./llama3-lora-out",
    per_device_train_batch_size=2,      # micro-batch that fits in VRAM
    gradient_accumulation_steps=8,      # effective batch = 2 * 8 = 16
    num_train_epochs=3,                 # 1-3 epochs is the sweet spot for SFT
    learning_rate=2e-4,                 # typical LR for LoRA fine-tuning
    lr_scheduler_type="cosine",         # decay LR smoothly toward the end
    warmup_ratio=0.03,                  # warm up LR over the first 3% of steps
    logging_steps=10,                   # log loss every 10 steps to watch the curve
    eval_strategy="epoch",              # evaluate on the val set each epoch
    bf16=True,                          # use bfloat16 mixed precision
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,             # tokenized + chat-templated in Week 3 Lesson 4
    eval_dataset=eval_ds,
    peft_config=lora_config,
)

trainer.train()
